In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%cd "C:/Users/mateo/Desktop/polymath/joint training experiments/algorithms/toys"

from toy_algorithms import (
    HIGH_DIM,
    LATENT_DIM,
    TOY_NAMES,
    TOY_CLASSES,
    device,
    make_toy,
    train_algorithm1,
    train_algorithm2,
    train_algorithm3,
    train_algorithm4,
    train_algorithm5,
    evaluate_toy_model,
    plot_toy_losses,
    plot_toy_analysis,
    print_toy_metrics,
)

print("Using:", device)


In [ ]:
ALGORITHM_NUMBER = 5
ALGORITHM_NAME = "Algorithm 5"
TRAIN_FUNCTION = train_algorithm5

NUM_EPOCHS = 50
TRAIN_SAMPLES = 3000
EVAL_SAMPLES = 2000
BATCH_SIZE = 256
NUM_LANDMARKS = 128
TEMPERATURE = 1.0

LAMBDA_KL = 1e-5
LAMBDA_DRIFT = 30.0
LAMBDA_VAR = 0.1
LAMBDA_COV = 0.3
LAMBDA_ADV = 1.0
LAMBDA_L1 = 0.0
VAE_PRETRAIN_EPOCHS = 25

EXTRA_KWARGS = {"lambda_l1": LAMBDA_L1, "vae_pretrain_epochs": VAE_PRETRAIN_EPOCHS}


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, toy_name in zip(axes, TOY_NAMES):
    _, labels, points = make_toy(toy_name, 2000, seed=7)
    axis.scatter(points[:, 0], points[:, 1], c=labels, s=4, cmap="tab10")
    axis.set_title(f"{toy_name}: underlying 2-D structure")
    axis.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"All toy observations have dimension {HIGH_DIM}.")
print("Temperature:", TEMPERATURE)


In [ ]:
models = {}
histories = {}
evaluations = {}

for toy_name in TOY_NAMES:
    print(f"\n===== {ALGORITHM_NAME}: {toy_name} =====")
    train_x, train_y, _ = make_toy(
        toy_name,
        TRAIN_SAMPLES,
        seed=7,
        high_dim=HIGH_DIM,
    )
    model, history = TRAIN_FUNCTION(
        train_x,
        train_y,
        TOY_CLASSES[toy_name],
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        latent_dim=LATENT_DIM,
        num_landmarks=NUM_LANDMARKS,
        T=TEMPERATURE,
        lambda_kl=LAMBDA_KL,
        lambda_drift=LAMBDA_DRIFT,
        lambda_var=LAMBDA_VAR,
        lambda_cov=LAMBDA_COV,
        lr=1e-3,
        **EXTRA_KWARGS,
    )
    eval_x, eval_y, _ = make_toy(
        toy_name,
        EVAL_SAMPLES,
        seed=700,
        high_dim=HIGH_DIM,
    )
    models[toy_name] = model
    histories[toy_name] = history
    evaluations[toy_name] = evaluate_toy_model(
        model,
        eval_x,
        eval_y,
        TOY_CLASSES[toy_name],
        seed=123,
    )


In [ ]:
# No image metrics or FID are used for the toy models.
# These summaries focus on distribution alignment and latent structure.
print_toy_metrics(evaluations)


In [ ]:
plot_toy_losses(histories, ALGORITHM_NAME)


In [ ]:
for toy_name in TOY_NAMES:
    plot_toy_analysis(
        evaluations[toy_name],
        toy_name,
        plot_samples=1000,
    )
